# Fit of transient photoluminescence (TrPL) with diffusion and multi trap recombination model
This example demonstrates how to fit transient photoluminescence (TrPL) using a rate equation model including recombination and diffusion effect using the `optimpv` package.

The rate equation and model used here are based on the work by [M. Simmonds](https://github.com/MaximSimmonds-HZB/MAPI-FAPI-fitting).  

Note: This is not ready for use yet!

In [ ]:
# Import necessary libraries
import warnings, os, sys, shutil
# remove warnings from the output
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from numpy.random import default_rng
import torch, copy, uuid
import pySIMsalabim as sim
from pySIMsalabim.experiments.JV_steady_state import *
import ax, logging
from ax.utils.notebook.plotting import init_notebook_plotting, render
init_notebook_plotting() # for Jupyter notebooks

try:
    from optimpv import *
    from optimpv.RateEqfits.RateEqAgent import RateEqAgent
    from optimpv.RateEqfits.RateEqModel import *
    from optimpv.RateEqfits.Pumps import *
except Exception as e:
    sys.path.append('../') # add the path to the optimpv module
    from optimpv import *
    from optimpv.RateEqfits.RateEqAgent import RateEqAgent
    from optimpv.RateEqfits.RateEqModel import *
    from optimpv.RateEqfits.Pumps import *

## Define the parameters for the simulation

In [ ]:
# Define the parameters to be fitted
params = []

Eg = FitParam(name = 'Eg', value = 1.6, bounds = [0.5,2.0], log_scale = False, rescale = True, value_type = 'float', type='fixed', display_name=r'$E_g$', unit='eV', axis_type = 'linear')
params.append(Eg)

L = FitParam(name = 'L', value = 450e-9, bounds = [400e-9,1e-6], log_scale = True, rescale = True, value_type = 'float', type='fixed', display_name=r'$L$', unit='m', axis_type = 'linear',force_log=True)
params.append(L)

alpha = FitParam(name = 'alpha', value = 64348.30886337494 *1e2, bounds = [1e6,1e8], log_scale = True, rescale = True, value_type = 'float', type='fixed', display_name=r'$\alpha$', unit='m$^{-1}$', axis_type = 'log',)
params.append(alpha)

N_cv = FitParam(name = 'N_cv', value = 2e24, bounds = [1e19,1e25], log_scale = True, rescale = True, value_type = 'float', type='fixed', display_name=r'$N_{cv}$', unit='m$^{-3}$', axis_type = 'log',force_log=True)
params.append(N_cv)

k_direct = FitParam(name = 'k_direct', value = 7.5e-17, bounds = [1e-18,1e-15], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$k_{\text{direct}}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log',force_log=True)
params.append(k_direct)

mu = FitParam(name = 'mu', value = 1e-10, bounds = [1e-8,1e-2], log_scale = True, rescale = True, value_type = 'float', type='fixed', display_name=r'$\mu$', unit='m$^{2}$ V$^{-1}$ s$^{-1}$', axis_type = 'log',force_log=True)
params.append(mu) # 4e-1*1e-4

N_t_bulk_1 = FitParam(name = 'N_t_bulk_1', value = 1e22, bounds = [1e19,1e25], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$N_{t,\text{bulk}}$', unit='m$^{-3}$', axis_type = 'log',force_log=True)
params.append(N_t_bulk_1)

C_n_1 = FitParam(name = 'C_n_1', value = 1e-14, bounds = [1e-20,1e-10], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$C_{n,1}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log',force_log=True)
params.append(C_n_1)

C_p_1 = FitParam(name = 'C_p_1', value = 1e-18, bounds = [1e-20,1e-10], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$C_{p,1}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log',force_log=True)
params.append(C_p_1)

E_t_bulk_1 = FitParam(name = 'E_t_bulk_1', value = 0.5, bounds = [0.2,Eg.value-0.2], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$E_{t,\text{bulk}}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log',force_log=True)
params.append(E_t_bulk_1)

N_t_bulk_2 = FitParam(name = 'N_t_bulk_2', value = 0e20, bounds = [1e19,1e25], log_scale = True, rescale = True, value_type = 'float', type='fixed', display_name=r'$N_{t,\text{bulk}}$', unit='m$^{-3}$', axis_type = 'log',force_log=True)
params.append(N_t_bulk_2)

C_n_2 = FitParam(name = 'C_n_2', value = 1e-17, bounds = [1e-20,1e-10], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$C_{n,2}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log',force_log=True)
params.append(C_n_2)

C_p_2 = FitParam(name = 'C_p_2', value = 1e-17, bounds = [1e-20,1e-10], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$C_{p,2}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log',force_log=True)
params.append(C_p_2)

E_t_bulk_2 = FitParam(name = 'E_t_bulk_2', value = 0.5, bounds = [0.2,Eg.value-0.2], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$E_{t,\text{bulk}}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log',force_log=True)
params.append(E_t_bulk_2)


I_factor_PL = FitParam(name = 'I_factor_PL', value = 1e-27, bounds = [1e-28,1e-26], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$I_{\text{PL}}$', unit='-', axis_type = 'log', force_log=True)
params.append(I_factor_PL)


# original values
params_orig = copy.deepcopy(params)
num_free_params = 0
dum_dic = {}
for i in range(len(params)):
    if params[i].force_log:
        dum_dic[params[i].name] = np.log10(params[i].value)
    else:
        dum_dic[params[i].name] = params[i].value/params[i].fscale
   # we need this just to run the model to generate some fake data

    if params[i].type != 'fixed'    :
        num_free_params += 1

In [ ]:
# import Data
parent_dir = os.path.abspath('../') # path to the parent directory if not in Notebooks use os.getcwd()
path2data  = os.path.join(parent_dir,'Data','perovskite_trPL')
data_raw = pd.read_csv(os.path.join(path2data,'Seo_FAPI1_Glass_0.dat'), sep=r'\s+',names=['t',"trPL"], skiprows=1)
# convert time to seconds
data_raw['t'] = data_raw['t'] * 1e-9 # convert to seconds
max_idx = data_raw['trPL'].idxmax()
# remove everything before the maximum
data_raw = data_raw.iloc[max_idx:]
# reset the index
data_raw = data_raw.dropna() # remove rows with negative trPL values
data_raw = data_raw[data_raw['trPL'] >= 0]# remove rows with negative trPL values
data_raw = data_raw.reset_index(drop=True)

# interpolate the data to have a logarithmically spaced time axis
t_log = np.logspace(np.log10(2e-9), np.log10(data_raw['t'].max()), num=100)
t_log = np.insert(t_log, 0, 0)  # add 0 to the time array
# interpolate the trPL values
trPL_log = np.interp(t_log, data_raw['t'] - data_raw['t'].min(), data_raw['trPL'])
data2fit = pd.DataFrame({'t': t_log - t_log.min(), 'trPL': trPL_log})

# plot the data
plt.figure()
plt.plot(data_raw['t'] - data_raw['t'].min(), data_raw['trPL'], 'o', label='Seo_FAPI1_Glass_0')
plt.plot(data2fit['t'], data2fit['trPL'], '*', label='Interpolated trPL')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Time [s]')
plt.ylabel('trPL [a.u.]')
plt.legend()
plt.show()

In [ ]:

time = data2fit['t'].values # time in seconds
X = time
y = data2fit['trPL'] 

# X = data_raw['t'].values - data_raw['t'].min() # time in seconds
# y = data_raw['trPL'] # dummy data

fpu = 50e3 # Frequency of the pump laser in Hz
Fluence = 4.8e15 # Fluence in m-2
z_array = np.linspace(0, L.value, 100) # z-axis in meters
generation = np.exp(-alpha.value * z_array)
generation_sum = np.trapezoid(generation, z_array)
n_0z = Fluence / generation_sum * generation
N0 = np.mean(n_0z) # mean initial carrier density in m-3
# print(N0)
# N0 = 2.88e13
background = 0e28 # Background illumination 


# Define the Agent and the target metric/loss function
metric = 'nrmse'
loss = 'linear' # 'nrmse' or 'mse' or 'soft_l1' or 'linear'
pump_args = {'N0': N0, 'fpu': fpu , 'background' : background, }
exp_format = 'trPL' # experiment format
RateEq = RateEqAgent(params, [X], [y], model = DBTD_multi_trap, pump_model = initial_carrier_density, pump_args = pump_args, fixed_model_args = {}, metric = metric, loss = loss,minimize=True,exp_format=exp_format,detection_limit=0e-5,  compare_type ='log')
y_test = RateEq.run({},exp_format=exp_format)

plt.figure()

# plt.plot(data2fit['t'], data2fit['trPL'], 'o', label='Seo_FAPI1_Glass_0')

plt.plot( data_raw['t'].values - data_raw['t'].min(), data_raw['trPL'].values, 'o', label='Seo_FAPI1_Glass_0 original')
plt.plot(X, y_test)
# plt.xlim(2e-8, 5e-5)
# plt.ylim(1e-21, 1e1)
# plt.xscale('log')
plt.yscale('log')
plt.xlabel('Time [s]')
plt.ylabel(exp_format + ' [a.u.]')
plt.show()
# print

## Run the optimization

In [ ]:
from optimpv.axBOtorch.axBOtorchOptimizer import axBOtorchOptimizer
from botorch.acquisition.logei import qLogNoisyExpectedImprovement 
from ax.adapter.transforms.standardize_y import StandardizeY
from ax.adapter.transforms.unit_x import UnitX
from ax.adapter.transforms.remove_fixed import RemoveFixed
from ax.adapter.transforms.log import Log
from ax.generators.torch.botorch_modular.utils import ModelConfig
from ax.generators.torch.botorch_modular.surrogate import SurrogateSpec
from gpytorch.kernels import MaternKernel
from gpytorch.kernels import ScaleKernel
from botorch.models import SingleTaskGP

model_gen_kwargs_list = None
parameter_constraints = None

model_kwargs_list = [{},{"torch_device":torch.device("cuda" if torch.cuda.is_available() else "cpu"),'botorch_acqf_class':qLogNoisyExpectedImprovement,'transforms':[RemoveFixed, Log,UnitX, StandardizeY],'surrogate_spec':SurrogateSpec(model_configs=[ModelConfig(botorch_model_class=SingleTaskGP,covar_module_class=ScaleKernel, covar_module_options={'base_kernel':MaternKernel(nu=2.5, ard_num_dims=num_free_params)})])}]

optimizer = axBOtorchOptimizer(params = params, agents = RateEq, models = ['SOBOL','BOTORCH_MODULAR'],n_batches = [1,40], batch_size = [8,2], ax_client = None,  max_parallelism = 100, model_kwargs_list = model_kwargs_list, model_gen_kwargs_list = model_gen_kwargs_list, name = 'ax_opti',parameter_constraints = parameter_constraints,parallel_agents= True)

In [ ]:
# optimizer.optimize() # run the optimization with ax
optimizer.optimize_turbo(force_continue=True,kwargs_turbo_state={'failure_tolerance':10}) # run the optimization with turbo

In [ ]:
# get the best parameters and update the params list in the optimizer and the agent
ax_client = optimizer.ax_client # get the ax client
optimizer.update_params_with_best_balance() # update the params list in the optimizer with the best parameters
RateEq.params = optimizer.params # update the params list in the agent with the best parameters

# print the best parameters
print('Best parameters:')
for p,po in zip(optimizer.params, params_orig):
    # print(p.name, 'fitted value:', p.value, 'original value:', po.value)
    print(f'{p.name} = {p.value:.2e} {p.unit} (original: {po.value:.2e} {po.unit}), bounds: [{p.bounds[0]:.2e}, {p.bounds[1]:.2e}] {p.unit}')
    

In [ ]:
# Plot optimization results
data = ax_client.summarize()
all_metrics = optimizer.all_metrics
plt.figure()
plt.plot(np.minimum.accumulate(data[all_metrics]), label="Best value seen so far")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Target metric: " + metric + " with " + loss + " loss")
plt.legend()
plt.title("Best value seen so far")

print("Best value seen so far is ", min(data[all_metrics[0]]), "at iteration ", int(data[all_metrics[0]].idxmin()))
plt.show()

In [ ]:
# rerun the simulation with the best parameters
yfit = RateEq.run(parameters={},exp_format=exp_format) # run the simulation with the best parameters

# viridis = plt.get_cmap('viridis', len(Gfracs))
plt.figure(figsize=(10,10))
linewidth = 2
plt.plot(X, yfit, label='Best fit', color='black', linewidth=linewidth)
plt.plot(X, y, 'o',label='Fitted data', color='blue', linewidth=linewidth)  
# for idx, Gfrac in enumerate(Gfracs[::-1]):
#     plt.plot(X[X[:,1]==Gfrac,0],y[X[:,1]==Gfrac],label='Gfrac = '+str(Gfrac),color=viridis(idx),alpha=0.5,linewidth=linewidth)
#     plt.plot(X[X[:,1]==Gfrac,0],yfit[X[:,1]==Gfrac],label='Gfrac = '+str(Gfrac)+' fit',linestyle='--',color=viridis(idx),linewidth=linewidth)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Time [s]')
plt.ylabel(exp_format + ' [a.u.]')
plt.legend()
plt.show()

In [ ]:
# from copy import deepcopy
# params_scipy = deepcopy(RateEq.params) # make copies of the parameters
# RateEq2 = deepcopy(RateEq) # make a copy of the agent
# from optimpv.scipyOpti.scipyOptimizer import ScipyOptimizer
# scipyOpti = ScipyOptimizer(params=params_scipy, agents=RateEq2, method='L-BFGS-B', options={'maxiter':200}, name='scipy_opti', parallel_agents=True, max_parallelism=os.cpu_count()-1, verbose_logging=True, )

In [ ]:
# scipyOpti.optimize() # run the optimization with scipy

In [ ]:
# # get the best parameters and update the params list in the optimizer and the agent
# RateEq2.params = scipyOpti.params # update the params list in the agent with the best parameters.append
# # print the best parameters
# # print the best parameters
# print('Best parameters:')
# for p,po in zip(scipyOpti.params, params_orig):
#     # print(p.name, 'fitted value:', p.value, 'original value:', po.value)
#     print(f'{p.name} = {p.value:.2e} {p.unit} (original: {po.value:.2e} {po.unit}), bounds: [{p.bounds[0]:.2e}, {p.bounds[1]:.2e}] {p.unit}')
    

In [ ]:
# # rerun the simulation with the best parameters
# yfit = RateEq2.run(parameters={},exp_format=exp_format) # run the simulation with the best parameters

# # viridis = plt.get_cmap('viridis', len(Gfracs))
# plt.figure(figsize=(10,10))
# linewidth = 2
# plt.plot(X, yfit, label='Best fit', color='black', linewidth=linewidth)
# plt.plot(X, y, 'o',label='Fitted data', color='blue', linewidth=linewidth)  

# plt.xscale('log')
# plt.yscale('log')
# plt.xlabel('Time [s]')
# plt.ylabel(exp_format + ' [a.u.]')
# plt.legend()
# plt.show()